# Notebook 02: Modeling & Evaluation


## **Intelligent Loan Pricing & Risk Management for Australian Banks**
### **Author: Pranamya Rajbhandari**

## **1. Import Python Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, roc_curve, auc)
from imblearn.over_sampling import SMOTE
import pickle
import warnings
warnings.filterwarnings('ignore')

## **2. Load Engineered Data**

In [ ]:
df_model = pd.read_csv('dataset\data_engineered_for_modeling.csv')

print("="*70)
print("DATA LOADED FOR MODELING")
print("="*70)
print(f"Shape: {df_model.shape}")
print(f"Columns: {df_model.columns.tolist()[:10]}... (showing first 10)")
print(f"\nTarget variable distribution:")
print(df_model['loan_approved'].value_counts())
print(f"Approval rate: {(df_model['loan_approved'] == 1).sum() / len(df_model) * 100:.1f}%")

df_model = df_model.drop('application_date', axis=1) #drop application_date as it is not needed for modeling



## **3. Prepare Data for Modeling**

In [ ]:
drop_cols = [
    'loan_approved'
]

X = df_model.drop(drop_cols, axis=1)
y = df_model['loan_approved']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
scaler.fit(X)
scaler.fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Train approval rate: {(y_train == 1).sum() / len(y_train) * 100:.1f}%")
print(f"Test approval rate: {(y_test == 1).sum() / len(y_test) * 100:.1f}%")


## **4. Handle Class Imbalance With SMOTE**

In [ ]:
# Ensure target is discrete (0 and 1)
y_train = y_train.astype(int)
y_test = y_test.astype(int)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"Balanced training set: {X_train_balanced.shape}")
print(f"Class distribution:")
print(pd.Series(y_train_balanced).value_counts())
print(f"Balanced approval rate: {(y_train_balanced == 1).sum() / len(y_train_balanced) * 100:.1f}%")


## **5. Train Baseline Models**

In [ ]:

print("TRAINING MODELS")


models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth = 12, min_samples_leaf = 5, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

trained_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_balanced, y_train_balanced)
    trained_models[name] = model
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    predictions[name] = y_pred
    probabilities[name] = y_pred_proba
    
    print(f"  - Fitted on {X_train_balanced.shape[0]} samples")

## **6. Evaluate Models**

In [ ]:
print("MODEL EVALUATION")
results = {}

for name, model in trained_models.items():
    y_pred = predictions[name]
    y_pred_proba = probabilities[name]
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'AUC-ROC': roc_auc
    }
    
    print(f"\n{name}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  AUC-ROC:   {roc_auc:.4f}")

results_df = pd.DataFrame(results).T
print("\n" + "="*70)
print("MODEL COMPARISON")
print(results_df)

In [ ]:
print("loan_approved" in X.columns)

In [ ]:
corr = df_model.corr(numeric_only=True)['loan_approved'].sort_values(key=abs, ascending=False)
print(corr.head(20))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions['Logistic Regression'])
print(cm)

In [ ]:
print(df_model['loan_approved'].value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print(scores)
print(scores.mean())